In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# OUTPUT / MULTI-SEED CONFIG
# ============================================================
N_SEEDS = 25
SEED_LIST = list(range(1, N_SEEDS + 1))   # 1..25
OUTPUT_DIR = "8cases_25seeds"
COMBINED_DIR = os.path.join(OUTPUT_DIR, "combined")


def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50, data=None):
    if data is None:
        data = qndata.stocks.load_data()

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns matrix
# ============================================================
def compute_returns_matrix(data, chosen_assets, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


# ============================================================
# 3) Rolling covariance
# ============================================================
def compute_sigma_series(R, lookback=252, annualize=252):
    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback + 1}.")

    Sigma_series = np.zeros((T, n, n))
    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t + 1, :]

        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        Sigma_series[t] = Sigma * annualize

    return Sigma_series


# ============================================================
# 4) Alpha models: simple vs AR(1)
# ============================================================
def mu_series_simple(R, lookback_mu=60, mode="momentum", annualize=252):
    T, n = R.shape
    mu = np.zeros((T, n))

    for t in range(T):
        start = max(0, t - lookback_mu + 1)
        window = R[start:t + 1, :]
        m = np.nanmean(window, axis=0)

        if mode == "mean_reversion":
            m = -m

        mu[t] = m * annualize

    return mu


def mu_series_ar1(R, lookback_model=252, annualize=252, smooth_alpha=0.20):
    T, n = R.shape
    mu_raw = np.zeros((T, n))

    for t in range(T):
        start = max(1, t - lookback_model + 1)
        window = R[start:t + 1, :]

        if window.shape[0] < 3:
            mu_raw[t] = 0.0
            continue

        X = window[:-1, :]
        Y = window[1:, :]

        for i in range(n):
            x = X[:, i]
            y = Y[:, i]
            A = np.column_stack([np.ones_like(x), x])

            try:
                theta, *_ = np.linalg.lstsq(A, y, rcond=None)
                a, b = theta
                mu_raw[t, i] = (a + b * R[t, i]) * annualize
            except Exception:
                mu_raw[t, i] = 0.0

    if smooth_alpha is None or smooth_alpha <= 0.0:
        return mu_raw

    mu_sm = np.zeros_like(mu_raw)
    mu_sm[0] = mu_raw[0]

    for t in range(1, T):
        mu_sm[t] = smooth_alpha * mu_raw[t] + (1.0 - smooth_alpha) * mu_sm[t - 1]

    return mu_sm


# ============================================================
# 5) Time-varying cost series
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t + 1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 6) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            s2 = w.sum()
            if s2 <= 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s2
    else:
        w = w / s

    return w


def turnover_from_weights(W):
    if W.shape[0] < 2:
        return np.array([])
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 7) Basic + One-Step
# ============================================================
def solve_basic_one_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)
    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
    )

    constraints = add_constraints(w, long_only=True, w_max=w_max, fully_invested=True)
    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 8) Basic + Two-Step
# ============================================================
def solve_basic_two_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    rho=25.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    # Step 1: target
    w_star = cp.Variable(n)
    objective1 = cp.Maximize(
        mu @ w_star - gamma * cp.quad_form(w_star, S)
    )
    constraints1 = add_constraints(w_star, long_only=True, w_max=w_max, fully_invested=True)
    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    # Step 2: trade towards target
    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
    )
    constraints2 = add_constraints(w, long_only=True, w_max=w_max, fully_invested=True)
    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 9) Enhanced + One-Step
# ============================================================
def solve_enhanced_one_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    gamma=5.0,
    eta=2.0,
    lam_t=0.0020,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)
    S = make_psd_matrix(Sigma, ridge=ridge)

    trade = w - w_prev
    abs_risk = cp.quad_form(w, S)
    te_risk = cp.quad_form(w - w_bench, S)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty = cp.norm2(D @ w)

    objective = cp.Maximize(
        mu @ w
        - gamma * abs_risk
        - eta * te_risk
        - lam_t * cp.norm1(trade)
        - kappa * robust_penalty
        - 1.0 * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 10) Enhanced + Two-Step
# ============================================================
def solve_enhanced_two_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    lam_t=0.0020,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    # Step 1: advanced target
    w_star = cp.Variable(n)

    abs_risk_star = cp.quad_form(w_star, S)
    te_risk_star = cp.quad_form(w_star - w_bench, S)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty_star = cp.norm2(D @ w_star)

    objective1 = cp.Maximize(
        mu @ w_star
        - gamma * abs_risk_star
        - eta * te_risk_star
        - kappa * robust_penalty_star
    )
    constraints1 = add_constraints(w_star, long_only=True, w_max=w_max, fully_invested=True)
    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    # Step 2: trade towards target with turnover control
    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
        + 1.0 * cp.sum_squares(trade)
    )
    constraints2 = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 11) Run one combination through time
# ============================================================
def run_strategy_combo(
    mu_series,
    Sigma_series,
    cost_series,
    family="basic",       # basic / enhanced
    method="one",         # one / two
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    w_bench = np.ones(n) / n

    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]
        lam_t = cost_series[t]

        if family == "basic" and method == "one":
            w_t = solve_basic_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "basic" and method == "two":
            w_star_t, w_t = solve_basic_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                rho=rho,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        elif family == "enhanced" and method == "one":
            w_t = solve_enhanced_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                w_bench=w_bench,
                gamma=gamma,
                eta=eta,
                lam_t=lam_t,
                kappa=kappa,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "enhanced" and method == "two":
            w_star_t, w_t = solve_enhanced_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                w_bench=w_bench,
                gamma=gamma,
                eta=eta,
                rho=rho,
                lam_t=lam_t,
                kappa=kappa,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        else:
            raise ValueError("Invalid family/method combination.")

        W[t] = w_t
        w_prev = w_t

    return W, W_star


# ============================================================
# 12) Realised returns
# ============================================================
def realised_returns_gross(R, W):
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 13) Summarise one strategy
# ============================================================
def summarise_strategy(name, W, W_star, R_daily, cost_series):
    to = turnover_from_weights(W)
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    return {
        "name": name,
        "W": W,
        "W_star": W_star,
        "turnover": to,
        "rp_gross": rp_gross,
        "rp_net": rp_net,
        "tc_paid": tc_paid,
        "eq_gross": eq_gross,
        "eq_net": eq_net,
        "stats_gross": stats_gross,
        "stats_net": stats_net,
        "avg_turnover": np.mean(to) if len(to) > 0 else np.nan,
        "max_turnover": np.max(to) if len(to) > 0 else np.nan,
        "avg_max_weight": float(np.mean(np.max(W, axis=1))) if W.size > 0 else np.nan,
        "has_nan_weights": bool(np.isnan(W).any()),
        "avg_cost_coeff": float(np.mean(cost_series)),
        "avg_realised_tc": float(np.mean(tc_paid)) if len(tc_paid) > 0 else np.nan,
    }


# ============================================================
# 14) Plot helpers
# ============================================================
def plot_grouped_results(times, results, cost_series):
    date_full = pd.to_datetime(times)
    date_idx = pd.to_datetime(times[1:])

    plt.figure(figsize=(10, 5))
    plt.plot(date_full, cost_series)
    plt.title("Time-varying transaction cost coefficient")
    plt.xlabel("Date")
    plt.ylabel("Cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["eq_net"], label=r["name"])
    plt.title("Net equity curves")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["turnover"], label=r["name"])
    plt.title("Turnover through time")
    plt.xlabel("Date")
    plt.ylabel("Turnover")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["tc_paid"], label=r["name"])
    plt.title("Realised trading costs through time")
    plt.xlabel("Date")
    plt.ylabel("Trading cost")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_heatmaps_for_results(results, max_plots=None):
    count = 0
    for r in results:
        if max_plots is not None and count >= max_plots:
            break

        plt.figure(figsize=(10, 4))
        plt.imshow(r["W"].T, aspect="auto")
        plt.title(f"Weights heatmap | {r['name']}")
        plt.xlabel("Time")
        plt.ylabel("Asset")
        plt.colorbar(label="Weight")
        plt.tight_layout()
        plt.show()

        if not np.isnan(r["W_star"]).all():
            plt.figure(figsize=(10, 4))
            plt.imshow(r["W_star"].T, aspect="auto")
            plt.title(f"Target weights heatmap | {r['name']}")
            plt.xlabel("Time")
            plt.ylabel("Asset")
            plt.colorbar(label="Target weight")
            plt.tight_layout()
            plt.show()

        count += 1


# ============================================================
# 15) CSV save helpers
# ============================================================
def strategy_slug(name):
    return (
        name.lower()
        .replace(" + ", "__")
        .replace("-", "_")
        .replace(" ", "_")
    )


def build_strategy_timeseries_df(times, result, seed, strategy_name):
    date_idx = pd.to_datetime(times[1:])

    return pd.DataFrame({
        "date": date_idx,
        "seed": seed,
        "strategy": strategy_name,
        "eq_gross": result["eq_gross"],
        "eq_net": result["eq_net"],
        "rp_gross": result["rp_gross"],
        "rp_net": result["rp_net"],
        "turnover": result["turnover"],
        "tc_paid": result["tc_paid"],
    })


def build_weights_df(times, W, chosen_assets, seed, strategy_name, value_name="weight"):
    df = pd.DataFrame(W, columns=chosen_assets)
    df.insert(0, "date", pd.to_datetime(times))
    df.insert(1, "seed", seed)
    df.insert(2, "strategy", strategy_name)
    df_long = df.melt(id_vars=["date", "seed", "strategy"], var_name="asset", value_name=value_name)
    return df_long


def save_seed_outputs(seed_dir, seed, chosen_assets, times, results, summary_df):
    # chosen assets
    chosen_assets_df = pd.DataFrame({
        "seed": seed,
        "asset": chosen_assets
    })
    chosen_assets_df.to_csv(os.path.join(seed_dir, "chosen_assets.csv"), index=False)

    # summary
    summary_df.to_csv(os.path.join(seed_dir, "seed_summary.csv"), index=False)

    all_seed_timeseries = []
    all_seed_weights = []
    all_seed_target_weights = []

    for r in results:
        slug = strategy_slug(r["name"])

        ts_df = build_strategy_timeseries_df(times, r, seed, r["name"])
        ts_df.to_csv(os.path.join(seed_dir, f"{slug}_timeseries.csv"), index=False)
        all_seed_timeseries.append(ts_df)

        w_df = build_weights_df(times, r["W"], chosen_assets, seed, r["name"], value_name="weight")
        w_df.to_csv(os.path.join(seed_dir, f"{slug}_weights.csv"), index=False)
        all_seed_weights.append(w_df)

        if not np.isnan(r["W_star"]).all():
            wstar_df = build_weights_df(times, r["W_star"], chosen_assets, seed, r["name"], value_name="target_weight")
            wstar_df.to_csv(os.path.join(seed_dir, f"{slug}_target_weights.csv"), index=False)
            all_seed_target_weights.append(wstar_df)

    seed_timeseries_df = pd.concat(all_seed_timeseries, ignore_index=True) if len(all_seed_timeseries) > 0 else pd.DataFrame()
    seed_weights_df = pd.concat(all_seed_weights, ignore_index=True) if len(all_seed_weights) > 0 else pd.DataFrame()
    seed_target_weights_df = pd.concat(all_seed_target_weights, ignore_index=True) if len(all_seed_target_weights) > 0 else pd.DataFrame()

    if len(seed_timeseries_df) > 0:
        seed_timeseries_df.to_csv(os.path.join(seed_dir, "all_strategies_timeseries.csv"), index=False)

    if len(seed_weights_df) > 0:
        seed_weights_df.to_csv(os.path.join(seed_dir, "all_strategies_weights.csv"), index=False)

    if len(seed_target_weights_df) > 0:
        seed_target_weights_df.to_csv(os.path.join(seed_dir, "all_strategies_target_weights.csv"), index=False)

    return {
        "chosen_assets_df": chosen_assets_df,
        "seed_summary_df": summary_df.copy(),
        "seed_timeseries_df": seed_timeseries_df,
        "seed_weights_df": seed_weights_df,
        "seed_target_weights_df": seed_target_weights_df,
    }


# ============================================================
# 16) Run one seed for 8 combinations
# ============================================================
def run_single_seed_all_comparisons(
    seed=7,
    n_assets=50,
    lookback_sigma=252,
    K_simple=60,
    ar_window=252,
    smooth_alpha=0.20,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualize=252,
    make_plots=True
):
    data = qndata.stocks.load_data()

    data, chosen_assets = load_quantiacs_universe(
        seed=seed,
        n_assets=n_assets,
        data=data
    )

    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    times, R_daily = compute_returns_matrix(
        data,
        chosen_assets,
        use_log_returns=use_log_returns
    )

    Sigma_series = compute_sigma_series(
        R_daily,
        lookback=lookback_sigma,
        annualize=annualize
    )

    mu_simple = mu_series_simple(
        R_daily,
        lookback_mu=K_simple,
        mode="momentum",
        annualize=annualize
    )

    mu_ar1 = mu_series_ar1(
        R_daily,
        lookback_model=ar_window,
        annualize=annualize,
        smooth_alpha=smooth_alpha
    )

    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    combo_specs = [
        ("Simple alpha + Basic + One-Step", mu_simple, "basic", "one"),
        ("Simple alpha + Basic + Two-Step", mu_simple, "basic", "two"),
        ("Simple alpha + Enhanced + One-Step", mu_simple, "enhanced", "one"),
        ("Simple alpha + Enhanced + Two-Step", mu_simple, "enhanced", "two"),
        ("AR1 alpha + Basic + One-Step", mu_ar1, "basic", "one"),
        ("AR1 alpha + Basic + Two-Step", mu_ar1, "basic", "two"),
        ("AR1 alpha + Enhanced + One-Step", mu_ar1, "enhanced", "one"),
        ("AR1 alpha + Enhanced + Two-Step", mu_ar1, "enhanced", "two"),
    ]

    results = []

    for name, mu_series, family, method in combo_specs:
        print(f"Running {name} ...")

        W, W_star = run_strategy_combo(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            family=family,
            method=method,
            gamma=gamma,
            eta=eta,
            rho=rho,
            kappa=kappa,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )

        res = summarise_strategy(
            name=name,
            W=W,
            W_star=W_star,
            R_daily=R_daily,
            cost_series=cost_series
        )
        results.append(res)

    rows = []
    for r in results:
        rows.append({
            "seed": seed,
            "strategy": r["name"],
            "final_equity_gross": r["eq_gross"][-1] if len(r["eq_gross"]) > 0 else np.nan,
            "final_equity_net": r["eq_net"][-1] if len(r["eq_net"]) > 0 else np.nan,
            "cum_return_gross": r["stats_gross"]["cum_return"],
            "cum_return_net": r["stats_net"]["cum_return"],
            "sharpe_gross": r["stats_gross"]["sharpe"],
            "sharpe_net": r["stats_net"]["sharpe"],
            "max_drawdown_gross": r["stats_gross"]["max_drawdown"],
            "max_drawdown_net": r["stats_net"]["max_drawdown"],
            "avg_turnover": r["avg_turnover"],
            "max_turnover": r["max_turnover"],
            "avg_max_weight": r["avg_max_weight"],
            "avg_cost_coeff": r["avg_cost_coeff"],
            "avg_realised_tc": r["avg_realised_tc"],
            "has_nan_weights": r["has_nan_weights"],
        })

    summary_df = pd.DataFrame(rows).sort_values(
        by=["sharpe_net", "final_equity_net"],
        ascending=False
    ).reset_index(drop=True)

    print("\n=== Summary table ===")
    print(summary_df)

    for r in results:
        print_stats(f"{r['name']} | Gross stats", r["stats_gross"])
        print_stats(f"{r['name']} | Net stats", r["stats_net"])

    if make_plots:
        plot_grouped_results(times, results, cost_series)
        plot_heatmaps_for_results(results, max_plots=8)

    return {
        "seed": seed,
        "chosen_assets": chosen_assets,
        "times": times,
        "R_daily": R_daily,
        "Sigma_series": Sigma_series,
        "mu_simple": mu_simple,
        "mu_ar1": mu_ar1,
        "cost_series": cost_series,
        "results": results,
        "summary_df": summary_df,
    }


# ============================================================
# 17) Run all 25 seeds and save CSV files
# ============================================================
def run_all_seeds_and_save(
    seeds=SEED_LIST,
    output_dir=OUTPUT_DIR,
    n_assets=50,
    lookback_sigma=252,
    K_simple=60,
    ar_window=252,
    smooth_alpha=0.20,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualize=252,
    make_plots=False
):
    safe_mkdir(output_dir)
    safe_mkdir(os.path.join(output_dir, "combined"))

    combined_summary_list = []
    combined_timeseries_list = []
    combined_weights_list = []
    combined_target_weights_list = []
    combined_chosen_assets_list = []

    for seed in seeds:
        print("\n" + "=" * 90)
        print(f"RUNNING SEED {seed}")
        print("=" * 90)

        seed_dir = os.path.join(output_dir, f"seed_{seed:03d}")
        safe_mkdir(seed_dir)

        out = run_single_seed_all_comparisons(
            seed=seed,
            n_assets=n_assets,
            lookback_sigma=lookback_sigma,
            K_simple=K_simple,
            ar_window=ar_window,
            smooth_alpha=smooth_alpha,
            use_log_returns=use_log_returns,
            gamma=gamma,
            eta=eta,
            rho=rho,
            kappa=kappa,
            tau=tau,
            w_max=w_max,
            ridge=ridge,
            base_cost=base_cost,
            cost_mode=cost_mode,
            vol_lookback=vol_lookback,
            alpha=alpha,
            annualize=annualize,
            make_plots=make_plots
        )

        saved = save_seed_outputs(
            seed_dir=seed_dir,
            seed=seed,
            chosen_assets=out["chosen_assets"],
            times=out["times"],
            results=out["results"],
            summary_df=out["summary_df"]
        )

        combined_summary_list.append(saved["seed_summary_df"])
        combined_chosen_assets_list.append(saved["chosen_assets_df"])

        if len(saved["seed_timeseries_df"]) > 0:
            combined_timeseries_list.append(saved["seed_timeseries_df"])

        if len(saved["seed_weights_df"]) > 0:
            combined_weights_list.append(saved["seed_weights_df"])

        if len(saved["seed_target_weights_df"]) > 0:
            combined_target_weights_list.append(saved["seed_target_weights_df"])

    combined_dir = os.path.join(output_dir, "combined")

    all_seed_summary = pd.concat(combined_summary_list, ignore_index=True)
    all_seed_summary.to_csv(os.path.join(combined_dir, "all_seed_summary.csv"), index=False)

    all_chosen_assets = pd.concat(combined_chosen_assets_list, ignore_index=True)
    all_chosen_assets.to_csv(os.path.join(combined_dir, "all_chosen_assets.csv"), index=False)

    if len(combined_timeseries_list) > 0:
        all_seed_timeseries = pd.concat(combined_timeseries_list, ignore_index=True)
        all_seed_timeseries.to_csv(os.path.join(combined_dir, "all_seed_timeseries.csv"), index=False)
    else:
        all_seed_timeseries = pd.DataFrame()

    if len(combined_weights_list) > 0:
        all_seed_weights = pd.concat(combined_weights_list, ignore_index=True)
        all_seed_weights.to_csv(os.path.join(combined_dir, "all_seed_weights.csv"), index=False)
    else:
        all_seed_weights = pd.DataFrame()

    if len(combined_target_weights_list) > 0:
        all_seed_target_weights = pd.concat(combined_target_weights_list, ignore_index=True)
        all_seed_target_weights.to_csv(os.path.join(combined_dir, "all_seed_target_weights.csv"), index=False)
    else:
        all_seed_target_weights = pd.DataFrame()

    avg_summary = (
        all_seed_summary
        .groupby("strategy", as_index=False)
        .agg({
            "final_equity_gross": "mean",
            "final_equity_net": "mean",
            "cum_return_gross": "mean",
            "cum_return_net": "mean",
            "sharpe_gross": "mean",
            "sharpe_net": "mean",
            "max_drawdown_gross": "mean",
            "max_drawdown_net": "mean",
            "avg_turnover": "mean",
            "max_turnover": "mean",
            "avg_max_weight": "mean",
            "avg_cost_coeff": "mean",
            "avg_realised_tc": "mean",
            "has_nan_weights": "max",
        })
        .sort_values(["sharpe_net", "final_equity_net"], ascending=False)
        .reset_index(drop=True)
    )

    avg_summary.to_csv(os.path.join(combined_dir, "average_metrics_across_seeds.csv"), index=False)

    print("\nSaved combined files:")
    print(os.path.join(combined_dir, "all_seed_summary.csv"))
    print(os.path.join(combined_dir, "all_chosen_assets.csv"))
    if len(all_seed_timeseries) > 0:
        print(os.path.join(combined_dir, "all_seed_timeseries.csv"))
    if len(all_seed_weights) > 0:
        print(os.path.join(combined_dir, "all_seed_weights.csv"))
    if len(all_seed_target_weights) > 0:
        print(os.path.join(combined_dir, "all_seed_target_weights.csv"))
    print(os.path.join(combined_dir, "average_metrics_across_seeds.csv"))

    return {
        "all_seed_summary": all_seed_summary,
        "all_chosen_assets": all_chosen_assets,
        "all_seed_timeseries": all_seed_timeseries,
        "all_seed_weights": all_seed_weights,
        "all_seed_target_weights": all_seed_target_weights,
        "average_metrics": avg_summary,
    }


# ============================================================
# 18) Main
# ============================================================
if __name__ == "__main__":
    combined_out = run_all_seeds_and_save(
        seeds=SEED_LIST,
        output_dir=OUTPUT_DIR,
        n_assets=50,
        lookback_sigma=252,
        K_simple=60,
        ar_window=252,
        smooth_alpha=0.20,
        use_log_returns=False,
        gamma=5.0,
        eta=2.0,
        rho=25.0,
        kappa=0.10,
        tau=0.05,
        w_max=0.10,
        ridge=1e-3,
        base_cost=0.0020,
        cost_mode="vol_scaled",   # or "constant"
        vol_lookback=21,
        alpha=2.0,
        annualize=252,
        make_plots=False          # set True only if you want plots for every seed
    )


RUNNING SEED 1


100% (367973 of 367973) |################| Elapsed Time: 0:00:00 Time:  0:00:00
100% (282488 of 282488) |################| Elapsed Time: 0:00:00 Time:  0:00:00
100% (10052928 of 10052928) |############| Elapsed Time: 0:00:00 Time:  0:00:00


fetched chunk 1/10 1s


100% (10052928 of 10052928) |############| Elapsed Time: 0:00:00 Time:  0:00:00


fetched chunk 2/10 3s


...

Running AR1 alpha + Basic + One-Step ...
Running AR1 alpha + Basic + Two-Step ...
Running AR1 alpha + Enhanced + One-Step ...
Running AR1 alpha + Enhanced + Two-Step ...

=== Summary table ===
   seed                            strategy  final_equity_gross  \
0    23     AR1 alpha + Enhanced + One-Step            3.699393   
1    23     AR1 alpha + Enhanced + Two-Step            3.162640   
2    23  Simple alpha + Enhanced + Two-Step            2.614852   
3    23  Simple alpha + Enhanced + One-Step            2.450173   
4    23        AR1 alpha + Basic + One-Step            3.139266   
5    23        AR1 alpha + Basic + Two-Step            3.131103   
6    23     Simple alpha + Basic + One-Step            2.272263   
7    23     Simple alpha + Basic + Two-Step            2.280017   

   final_equity_net  cum_return_gross  cum_return_net  sharpe_gross  \
0          3.346440          2.699393        2.346440      1.290975   
1          2.859367          2.162640        1.859367      1.

/usr/local/lib/python3.11/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Running AR1 alpha + Basic + One-Step ...
Running AR1 alpha + Basic + Two-Step ...
Running AR1 alpha + Enhanced + One-Step ...
Running AR1 alpha + Enhanced + Two-Step ...

=== Summary table ===
   seed                            strategy  final_equity_gross  \
0    24     AR1 alpha + Enhanced + Two-Step            2.303898   
1    24     AR1 alpha + Enhanced + One-Step            2.318850   
2    24  Simple alpha + Enhanced + Two-Step            1.768542   
3    24        AR1 alpha + Basic + One-Step            2.444943   
4    24        AR1 alpha + Basic + Two-Step            2.435652   
5    24  Simple alpha + Enhanced + One-Step            1.587172   
6    24     Simple alpha + Basic + One-Step            1.448638   
7    24     Simple alpha + Basic + Two-Step            1.444963   

   final_equity_net  cum_return_gross  cum_return_net  sharpe_gross  \
0          2.081962          1.303898        1.081962      1.043333   
1          2.097165          1.318850        1.097165      1.

/usr/local/lib/python3.11/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Running AR1 alpha + Basic + One-Step ...
Running AR1 alpha + Basic + Two-Step ...
Running AR1 alpha + Enhanced + One-Step ...
Running AR1 alpha + Enhanced + Two-Step ...

=== Summary table ===
   seed                            strategy  final_equity_gross  \
0    25     AR1 alpha + Enhanced + One-Step            2.946944   
1    25     AR1 alpha + Enhanced + Two-Step            2.710765   
2    25  Simple alpha + Enhanced + Two-Step            2.113961   
3    25        AR1 alpha + Basic + One-Step            3.092908   
4    25        AR1 alpha + Basic + Two-Step            3.105437   
5    25  Simple alpha + Enhanced + One-Step            1.709074   
6    25     Simple alpha + Basic + One-Step            1.851584   
7    25     Simple alpha + Basic + Two-Step            1.852271   

   final_equity_net  cum_return_gross  cum_return_net  sharpe_gross  \
0          2.664223          1.946944        1.664223      1.198665   
1          2.449066          1.710765        1.449066      1.